# Template for Multi-agent Knowledge Base System Built on Router Pattern

_Building a multi-agent system following router pattern._

Build a simple multi-agent system following router pattern where the router should be able to decompose the user query into source-specific sub-questions, route them to the relevant specialist agents in parallel, and synthesize results into a coherent answer. Use the below mentioned specialist agents.

- A GitHub Agent: Searches code, issues, and pull requests.
- A Notion Agent: Searches internal documentation and wikis.
- A Slack Agent: Searches relevant threads and discussions.

In [ ]:
# Imports packages

# Import class `ChatOllama` from module `langchain_ollama`
# Import class `Annotated`, `Literal`, `TypedDict` from module `typing`
# Import function `tool` from module `langchain.tools`
# Import function `create_agent` from module `langchain.agents`
# Import class `BaseModel` and function `Field` from module `pydantic`
# Import class `StateGraph` and constant `START`, `END` from module `langgraph.graph`
# Import class `Send` from module `langgraph.types`
# Import module `operator`

## Model

_Connecting an appropriate model to a chat client._

In [3]:
# Sets endpoints for Ollama models to be available over web requests.

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_ENDPOINT = "http://localhost:11434/"

In [ ]:
# Initialize a chat client by calling constuctor of class ChatOllama with the following arguments.
# 1. model name against parameter `model`, 
# 2. 'False` against parameter `reasoning`, and
# 3. model endpoint to parameter `base_url`, and
# store the reference to the created instance in a variable called `model`.

# Code here
# ...

## States
Defining states schemas.

In [5]:
class Classification(TypedDict):
    """A single routing decision: which agent to call with what query"""
    source: Literal["github", "notion", "slack"]
    query: str

class AgentInput(TypedDict):
    """Simple input state passed to each sub-agent."""
    query: str

class AgentOutput(TypedDict):
    """Result return by each sub-agent."""
    source: str
    result: str

class RouterState(TypedDict):
    """Main workflow state tracking the query, classifications, results, and final answer."""
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]     # Uses a reducer `operator.add` to collect outputs from
                                                            # parallel agent executions into a single list.
    final_answer: str

## Tools
Creates tools for each knowledge domain e.g. GitHub, Notion and Slack. 

For this experiment, dummy implementations that return mock data are used. In a production system, these would call actual APIs.

In [6]:
# ========== 3 TOOLS RELATED TO GITHUB ==========
@tool
def search_code(query: str, repo: str = "main") -> str:
    """Searches code in GitHub repositories."""
    return f"Found code matching '{query}' in {repo}: authentication middleware in src/auth.py"

@tool
def search_issues(query: str) -> str:
    """Searches GitHub issues."""
    return f"Found 3 issues matching '{query}': #142 (API auth docs), #89 (OAuth flow), #203 (token refresh)"

@tool
def search_prs(query: str) -> str:
    """Searches pull requests for implementation details."""
    return f"PR #156 added JWT authentication, PR #178 updated OAuth scopes"


# ========== 2 TOOLS RELATED TO NOTION ==========
@tool
def search_notion(query: str) -> str:
    """Searches Notion workspace for documentation."""
    return f"Found documentation: 'API Authentication Guide' - covers OAuth2 flow, API keys, and JWT tokens"


@tool
def get_page(page_id: str) -> str:
    """Get a specific Notion page by ID."""
    return f"Page content: Step-by-step authentication setup instructions"


# ========== 2 TOOLS RELATED TO SLACK ==========
@tool
def search_slack(query: str) -> str:
    """Searches Slack messages and threads."""
    return f"Found discussion in #engineering: 'Use Bearer tokens for API auth, see docs for refresh flow'"

@tool
def get_thread(thread_id: str) -> str:
    """Gets a specific Slack thread."""
    return f"Thread discusses best practices for API key rotation"

## Sub-Agents
_Create sub-agent for each vertical. Each sub-agent should have domain-specific tools and a prompt optimized for its knowledge source. All three should follow the same pattern — only the tools and system prompt will differ._

In [ ]:
GITHUB_PROMPT=(
        "You are a GitHub expert. Answer questions about code, "
        "API references, and implementation details by searching "
        "repositories, issues, and pull requests."
    )

# Create the GitHub sub-agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `search_code`, `search_issues` and `search_prs` against parameter `tools`, and
# 3. the GitHub agent prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `github_agent`.

# Code here
# ...

In [ ]:
NOTION_PROMPT=(
        "You are a Notion expert. Answer questions about internal "
        "processes, policies, and team documentation by searching "
        "the organization's Notion workspace."
    )

# Create the Notion sub-agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `search_notion` and `get_page` against parameter `tools`, and
# 3. the Notion agent prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `notion_agent`.

# Code here
# ...

In [ ]:
SLACK_PROMPT=(
        "You are a Slack expert. Answer questions by searching "
        "relevant threads and discussions where team members have "
        "shared knowledge and solutions."
    )

# Create the Slack sub-agent by calling function `create_agent` with the following arguments.
# 1. the model instance as the first (positioned) parameter,
# 2. a list containing tool function `search_slack` and `get_thread` against parameter `tools`, and
# 3. the Slack agent prompt against parameter `system_prompt`, and
# store the reference of the created instance into a variable called `slack_agent`.

# Code here
# ...

## Router Workflow Design
Build the router workflow using a state graph in `LangGraph`. The main steps in the workflow should be:

1. Classify: Analyzes the query and determines which agents to invoke with what sub-questions.
2. Route: Fans out to selected agents in parallel using `Send`.
3. Query Agents: Each agent receives a simple `AgentInput` and returns an `AgentOutput`.
4. Synthesize: Combines collected results into a coherent response.

In [10]:
SYSTEM_PROMPT = """Analyze this query and determine which knowledge bases to consult.
For each relevant source, generate a targeted sub-question optimized for that source.

Available sources:
- github: Code, API references, implementation details, issues, pull requests
- notion: Internal documentation, processes, policies, team wikis
- slack: Team discussions, informal knowledge sharing, recent conversations

Return ONLY the sources that are relevant to the query. Each source should have
a targeted sub-question optimized for that specific knowledge domain.

Example for "How do I authenticate API requests?":
- github: "What authentication code exists? Search for auth middleware, JWT handling"
- notion: "What authentication documentation exists? Look for API auth guides"
(slack omitted because it's not relevant for this technical question)"""

**Classification**

Analyzing the query and determining which agents to invoke with what sub-questions.

In [11]:
# Defines structured output schema for the classifier
class ClassificationResult(BaseModel):
    """Result of classifying a user query into agent-specific sub-questions."""
    classifications: list[Classification] = Field(
        description="List of agents to invoke with their targeted sub-questions"
    )

def classify_query(state: RouterState) -> dict:
    """Classifies query and determine which agents to invoke."""
    
    structured_model = model.with_structured_output(ClassificationResult)

    result = structured_model.invoke([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": state["query"]}
    ])

    return {"classifications": result.classifications}

**Routing**

Fanning out to selected agents in parallel using `Send`.

In [12]:
def route_to_agents(state: RouterState) -> list[Send]:
    """Fans out to agents based on classifications."""
    return [
        Send(c["source"], {"query": c["query"]}) for c in state["classifications"]
    ]

**Query Agents**

Each agent receiving a simple `AgentInput` and returing an `AgentOutput.`

In [13]:
def query_github(state: AgentInput) -> dict:
    """Queries the GitHub agent."""
    result = github_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "github", "result": result["messages"][-1].content}]}


def query_notion(state: AgentInput) -> dict:
    """Queries the Notion agent."""
    result = notion_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "notion", "result": result["messages"][-1].content}]}


def query_slack(state: AgentInput) -> dict:
    """Queries the Slack agent."""
    result = slack_agent.invoke({
        "messages": [{"role": "user", "content": state["query"]}]
    })
    return {"results": [{"source": "slack", "result": result["messages"][-1].content}]}

**Synthesizing**

Combining collected results into a coherent response.

In [14]:
SYNTHESIS_PROMPT="""Synthesize these search results to answer the original question: {query}

- Combine information from multiple sources without redundancy
- Highlight the most relevant and actionable information
- Note any discrepancies between sources
- Keep the response concise and well-organized"""

def synthesize_results(state: RouterState) -> dict:
    """Combines results from all agents into a coherent answer."""
    if not state["results"]:
        return {"final_answer": "No results found from any knowledge source."}

    # Format results for synthesis
    formatted = [
        f"**From {r['source'].title()}:**\n{r['result']}" for r in state["results"]
    ]

    synthesis_response = model.invoke([
        {
            "role": "system",
            "content": SYNTHESIS_PROMPT.format(query=state['query'])
        },
        {"role": "user", "content": "\n\n".join(formatted)}
    ])

    return {"final_answer": synthesis_response.content}

**Workflow Compilation**

Assembles the workflow by connecting nodes with edges. 

Using `add_conditional_edges` with the routing function enables parallel execution.

The `add_conditional_edges` call connects the classify node to the agent nodes through the `route_to_agents` function. When `route_to_agents` returns multiple `Send` objects, those nodes execute in parallel.

In [ ]:
# Instantiate state graph by calling constructor `StateGraph` with `RouterState` as argument against
# first positioned parameter and store the reference of created in a variable `builder`.

# Code here
# ...


# Adding classification node
# Call function `add_node` of the builder passing 
# value "classify" as argument against first positioned parameter and
# function pointer `classify_query` against the second positioned parameter

# Code here
# ...


# Adding sub-agent nodes
# Call function `add_node` of the builder passing 
# value "github" as argument against first positioned parameter and
# node function pointer `query_github` against the second positioned parameter
# Repeat the similar step to add nodes for notion and slack

# Code here
# ...

# Adding synthesizer node
# Call function `add_node` of the builder passing 
# value "synthesize" as argument against first positioned parameter and
# node function pointer `synthesize_results` against the second positioned parameter

# Code here
# ...


# Adding starting edge
# Call function `add_node` of the builder passing 
# constant `START` as argument against first positioned parameter and
# value "classify" against the second positioned parameter

# Code here
# ...


# Adding conditional edge
# Call function `add_conditional_edges` of the builder passing 
# value "classify" against the first positioned parameter,
# node function pointer `route_to_agents` against the second positioned parameter and
# a list containing values "github", "notion" and "slack" against the third positioned parameter.

# Code here
# ...


# Adding consolidation edges
# Call function `add_edge` of the builder to add edge from node "github" to "synthesize" passing
# value "github" as argument against first positioned parameter and
# value "synthesize" against the second positioned parameter.
# Repeat the similar steps to edge (1) from node "notion" to "synthesize" and (2) from node "slack" to "synthesize".

# Code here
# ...


# Adding termination edge
# Call function `add_edge` of the builder passing
# value "synthesize" as argument against first positioned parameter and
# constant `END` as argument against second positioned parameter.

# Code here
# ...


# Compile the graph to get the workflow by calling function `compile` of the builder without any arguments and
# store the reference of the compiled state graph in a variable `workflow`.

# Code here
# ...

## Testing Workflow
Test the router with queries that span multiple knowledge domains.

In [ ]:
# NOTE THAT THIS STEP MAY TAKE AROUND TWO MINUTES TO COMPLETE

result = workflow.invoke({
    "query": "How do I authenticate API requests?"
})

print("Original query:", result["query"])
print("\nClassifications:")
for c in result["classifications"]:
    print(f"  {c['source']}: {c['query']}")
print("\n" + "=" * 60 + "\n")
print("Final Answer:")
print(result["final_answer"])